# `semantic.v_horizon_performance` — view

**The 440 horizon rows, computed rather than stored.** Grain (ticker, horizon), five
horizons: 15, 10, 5, 3 and 1 year.

This used to be a table. Step 14 moved the fact down to its atomic grain — one row per
(ticker, month, manager) — which left the horizon numbers with nowhere to live but here.
The CTE chain below is the old `gold_etl_fact_horizon_performance` almost verbatim; what is
new is where it reads from, and the `series` step in front of it.

Three window functions do work that would otherwise need repeated aggregation:

| window | what it does |
|---|---|
| `SUM(LN(1+r)) OVER (ORDER BY month_key)` | SPY compounded at every month, in one pass. Adding logarithms is multiplying, and SQL has no running product |
| the same frame ending `1 PRECEDING` | the curve shifted one month, so any span becomes one point divided by another |
| `RANK() OVER (PARTITION BY horizon_years ORDER BY ...)` | three leaderboards — return, income, risk-adjusted — in the same pass |

The four views above this one read it instead of reading the fact. That is on purpose: it is
the **single place** the fan-out is collapsed, so no other view can get it wrong.

A view is its own definition, so there is no load step and no etl task to pair with this one.

In [ ]:
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_horizon_performance
COMMENT 'Return, income and risk for every ticker at five horizons. The answer, computed from the atomic fact'
AS
WITH series AS (
  -- THE MOST IMPORTANT LINE IN THIS VIEW. The fact carries one row per manager, so a
  -- six-manager trust holds six copies of every month. Collapse them BEFORE anything
  -- compounds, or every return in the project is inflated by the fan-out.
  SELECT DISTINCT ticker_key, ticker, management_group_key, mandate_key, month_key,
         close, dividend, price_return, total_return
  FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance
),
clock AS (
  -- dim_date supplies the date arithmetic; the fact carries only the YYYYMM key.
  SELECT MAX(d.month_start) AS as_of_start, MAX(d.month_key) AS as_of_key
  FROM series f
  JOIN `index-vs-trust-pipeline`.gold.dim_date d ON d.month_key = f.month_key
),
horizons AS (
  SELECT * FROM VALUES (15), (10), (5), (3), (1) AS h(horizon_years)
),
windows AS (
  SELECT h.horizon_years,
         -- The same bounds as YYYYMM, so the comparison filters on the fact's own key.
         CAST(DATE_FORMAT(ADD_MONTHS(c.as_of_start,
              -(h.horizon_years * 12) + 1), 'yyyyMM') AS INT)   AS win_start_key,
         c.as_of_key                                            AS win_end_key,
         c.as_of_key                                            AS as_of_key,
         h.horizon_years * 12                                   AS window_months
  FROM horizons h CROSS JOIN clock c
),
spy_curve AS (
  SELECT month_key,
         EXP(SUM(LN(1 + COALESCE(total_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS cum_total_incl,
         EXP(SUM(LN(1 + COALESCE(total_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)) AS cum_total_excl,
         EXP(SUM(LN(1 + COALESCE(price_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS cum_price_incl,
         EXP(SUM(LN(1 + COALESCE(price_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)) AS cum_price_excl
  FROM series
  WHERE ticker = 'SPY'
),
observations AS (
  -- Every side is a total return now, so there is one basis and no branch. The column is
  -- still carried, so the answer records how each return was built rather than assuming it.
  -- The group and mandate keys come off the fact, not off dim_ticker: the fact carries them.
  SELECT d.ticker_key, m.management_group_key, m.mandate_key,
         d.ticker, w.horizon_years, w.window_months, w.as_of_key,
         'total'         AS return_basis,
         m.total_return  AS r,
         m.price_return, m.month_key, m.close, m.dividend
  FROM `index-vs-trust-pipeline`.gold.dim_ticker d
  CROSS JOIN windows w
  JOIN series m
    ON m.ticker = d.ticker AND m.month_key BETWEEN w.win_start_key AND w.win_end_key
  WHERE d.is_current
    AND d.months_available >= 36
    AND m.total_return IS NOT NULL
),
aggregated AS (
  SELECT ticker_key, management_group_key, mandate_key, ticker, horizon_years,
         window_months, as_of_key, return_basis,
         COUNT(*)                AS months_used,
         MIN(month_key)          AS start_month_key,
         MAX(month_key)          AS end_month_key,
         EXP(SUM(LN(1 + r))) - 1 AS total_return,
         STDDEV(r) * SQRT(12)    AS volatility,
         SUM(dividend) / NULLIF(MIN_BY(close, month_key), 0) AS income_return,
         -- Growth alone: the same compounding, over price only. Dividends excluded.
         EXP(SUM(LN(1 + COALESCE(price_return, 0)))) - 1     AS price_return
  FROM observations
  GROUP BY ticker_key, management_group_key, mandate_key, ticker, horizon_years,
           window_months, as_of_key, return_basis
  HAVING COUNT(*) >= 0.8 * window_months
),
compared AS (
  SELECT a.*,
         e.cum_total_incl / s.cum_total_excl - 1               AS index_return_same_period,
         e.cum_price_incl / s.cum_price_excl - 1               AS index_price_return_same_period
  FROM aggregated a
  JOIN spy_curve s ON s.month_key = a.start_month_key
  JOIN spy_curve e ON e.month_key = a.end_month_key
),
index_risk AS (
  SELECT c.horizon_years, c.start_month_key, c.end_month_key,
         STDDEV(x.total_return) * SQRT(12) AS index_volatility_same_period
  FROM (SELECT DISTINCT horizon_years, start_month_key, end_month_key FROM compared) c
  JOIN series x
    ON x.ticker = 'SPY' AND x.month_key BETWEEN c.start_month_key AND c.end_month_key
  GROUP BY c.horizon_years, c.start_month_key, c.end_month_key
),
final AS (
  SELECT c.*, r.index_volatility_same_period
  FROM compared c
  JOIN index_risk r
    ON r.horizon_years = c.horizon_years
   AND r.start_month_key = c.start_month_key
   AND r.end_month_key = c.end_month_key
)
SELECT MD5(CONCAT_WS('|', ticker, CAST(horizon_years AS STRING)))      AS horizon_key,
       ticker_key, management_group_key, mandate_key,
       ticker, as_of_key AS month_key, horizon_years,
       start_month_key, end_month_key, months_used,
       total_return,
       POWER(1 + total_return, 12.0 / months_used) - 1                 AS annualised_return,
       income_return,
       volatility,
       index_return_same_period,
       index_volatility_same_period,
       -- A tolerance, not a bare '>'. SPY's own row compares it against itself: the two
       -- sides are the same quantity down different code paths, so floating-point SUM order
       -- can leave them 1 ULP apart and flip the flag between query plans. No trust is
       -- within 0.19 of the boundary, so this pins SPY to false and touches nothing else.
       total_return > index_return_same_period + 1e-9                  AS beat_index,
       (POWER(1 + total_return, 12.0 / months_used) - 1)
         / NULLIF(volatility, 0)                                       AS risk_adjusted_return,
       return_basis,
       RANK() OVER (PARTITION BY horizon_years ORDER BY total_return DESC)  AS rank_by_return,
       RANK() OVER (PARTITION BY horizon_years ORDER BY income_return DESC) AS rank_by_income,
       RANK() OVER (PARTITION BY horizon_years
                    ORDER BY (POWER(1 + total_return, 12.0 / months_used) - 1)
                             / NULLIF(volatility, 0) DESC)                  AS rank_by_risk_adjusted,
       price_return,
       index_price_return_same_period,
       price_return > index_price_return_same_period + 1e-9            AS out_grew_index
FROM final;

## Verification

Expected: **440 rows**, and `under_coverage` **0** at every horizon.

In [ ]:
SELECT horizon_years,
       COUNT(*)                                                 AS rows,
       SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)              AS beat_count,
       SUM(CASE WHEN out_grew_index THEN 1 ELSE 0 END)          AS out_grew_count,
       SUM(CASE WHEN months_used < 0.8 * horizon_years * 12
                THEN 1 ELSE 0 END)                              AS under_coverage,
       SUM(CASE WHEN price_return IS NULL THEN 1 ELSE 0 END)    AS null_growth
FROM `index-vs-trust-pipeline`.semantic.v_horizon_performance
GROUP BY horizon_years
ORDER BY horizon_years DESC;

Expect **440 rows** in total, `under_coverage` **0** and `null_growth` **0** at every horizon:

| horizon | rows | beat | out_grew |
|---|---|---|---|
| 15 | **78** | 6 | 6 |
| 10 | **86** | 11 | 9 |
| 5 | **92** | 15 | 8 |
| 3 | **92** | 28 | 21 |
| 1 | **92** | 39 | 32 |

These counts run over everything in the view, **index rows included**, so they are not the
beat rate. The trust counts are **75 / 83 / 89 / 89 / 89** and the beat rate is
**5.3 / 10.8 / 14.6 / 29.2 / 41.6**, which `v_beat_rate` computes with `entity_type = 'Trust'`.

### Why 3 and 1 year are one lower than the table they replaced

The retired `gold.fact_horizon_performance` records SPY as **beating itself** at 3 and 1
year, and as out-growing itself at 1 year. Its stored gaps are `+2.22e-16` — floating-point
noise, not performance. `EXP(SUM(LN(1+r)))` and the `spy_curve` ratio are the same quantity
down two different code paths, and Spark's `SUM` is order-dependent in the last bits, so the
flag could flip between query plans.

The `+ 1e-9` tolerance on `beat_index` pins it to false. **No trust is anywhere near that
boundary** — the closest, `ESCT` at 15 years, sits 0.19 away, eight orders of magnitude
out — so the tolerance changes SPY's three flags and nothing else at all.

### The guard that matters

The step that created this view is **not allowed to move a single number**. This cell is the
proof: it compares the view against the table it replaced, row for row and column for
column. Run it while `gold.fact_horizon_performance` still exists, and delete the table only
once it returns zeros.

If `total_return` disagrees anywhere, the `series` CTE has failed to collapse the fan-out
and every figure in the project is inflated.

In [ ]:
SELECT COUNT(*)                                                      AS rows_compared,
       SUM(CASE WHEN ABS(v.total_return - o.total_return) > 1e-9
                THEN 1 ELSE 0 END)                                   AS total_return_differs,
       SUM(CASE WHEN ABS(v.volatility  - o.volatility)  > 1e-9
                THEN 1 ELSE 0 END)                                   AS volatility_differs,
       SUM(CASE WHEN ABS(v.income_return - o.income_return) > 1e-9
                THEN 1 ELSE 0 END)                                   AS income_differs,
       SUM(CASE WHEN v.beat_index <> o.beat_index THEN 1 ELSE 0 END)  AS beat_flag_differs,
       SUM(CASE WHEN v.rank_by_return <> o.rank_by_return
                THEN 1 ELSE 0 END)                                    AS rank_differs,
       MAX(ABS(v.total_return - o.total_return))                      AS largest_gap
FROM `index-vs-trust-pipeline`.semantic.v_horizon_performance v
FULL OUTER JOIN `index-vs-trust-pipeline`.gold.fact_horizon_performance o
  ON o.ticker = v.ticker AND o.horizon_years = v.horizon_years;

Expect **440 / 0 / 0 / 0 / 2** with `largest_gap` around **4.6e-14**, and `which` naming
exactly **`SPY@1, SPY@3`**.

`rows_compared` must be exactly 440: a `FULL OUTER JOIN` is used rather than an inner one so
that a row present on only one side still shows up and inflates the count.

**Zero on `total_return`, `volatility` and `income_return` is the acceptance test for step
14** — the fact moved to atomic grain and the maths moved into a view, and not one measured
value changed.

The two flag differences are **the fix, not a failure**. Both are SPY judged against itself;
see the note above. `out_grew_index` differs on `SPY@1` for the same reason. Any difference
on a row that is not SPY means the `series` CTE has failed to collapse the fan-out, and every
return in the project is inflated.